In [ ]:
import torch
import os

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

def create_sample():
    sample = []
    jubaea_directory = "Data/Jubaea"
    for image in os.listdir(jubaea_directory):
        if image.lower().endswith((".jpg", ".jpeg", ".png")):
            sample.append((os.path.join(jubaea_directory, image),1))


    not_jubaea_directory = "Data/Not_Jubaea"
    for subfolder in os.listdir(not_jubaea_directory):
        for image in os.listdir(os.path.join(not_jubaea_directory, subfolder)):
            if image.lower().endswith((".jpg", ".jpeg", ".png")):
                sample.append((os.path.join(not_jubaea_directory,subfolder,image),0))
    return sample

sample = create_sample()
pictures = [x[0] for x in sample]
label = [x[1] for x in sample]
X_train, X_test, Y_train, Y_test = train_test_split(pictures, label, test_size=0.2, stratify=label)

transform = transforms.Compose([
transforms.Resize((128, 128)),transforms.ToTensor()
])

class JubaeaDataset(Dataset):
    def __init__(self, path, label, transform):
        self.path = path
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.path)

    def __getitem__(self, index):
        path = self.path[index]
        label = self.label[index]

        image = Image.open(path).convert("RGB")
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = transform(image)

        return image, label

train_dataset = JubaeaDataset(X_train, Y_train, transform=transform)
test_dataset = JubaeaDataset(X_test, Y_test, transform=transform)